In [1]:
import pandas as pd
import numpy as np
import polars as pl
import time

import timeit

In [2]:
def best_ms(fn, n=10):
    return min(timeit.repeat(fn, number=1, repeat=n)) * 1000

In [3]:
results = []

In [4]:
task = "late_delivery_flag"
print(f"Task: {task}")

# Pandas Apply Function
orders = pd.read_csv("../data/orders.csv", parse_dates=["order_ts"])
pandas_apply_ms = best_ms(
    lambda: orders.apply(
        lambda x: x["delivery_min"] > 15 if x["city"] in ["Delhi", "Mumbai", "Bengaluru"] else x["delivery_min"] > 20,
        axis=1,
    )
)
print(f"Time taken by pandas_apply: {pandas_apply_ms:.2f} ms")

# Pandas Vector Function
orders = pd.read_csv("../data/orders.csv", parse_dates=["order_ts"])
pandas_vector_ms = best_ms(
    lambda: orders["delivery_min"] > orders["city"].map({"Delhi": 15, "Mumbai": 15, "Bengaluru": 15}).fillna(20)
)
print(f"Time taken by pandas_vector: {pandas_vector_ms:.2f} ms")

# Numpy Function
orders = pd.read_csv("../data/orders.csv", parse_dates=["order_ts"])
numpy_ms = best_ms(
    lambda: np.where(orders["city"].isin(["Delhi", "Mumbai", "Bengaluru"]), orders["delivery_min"] > 15, orders["delivery_min"] > 20)
)
print(f"Time taken by numpy: {numpy_ms:.2f} ms")

# Polars
orders = pl.read_csv("../data/orders.csv", try_parse_dates=True)
polars_ms = best_ms(
    lambda: orders.with_columns(
        pl.when(pl.col("city").is_in(["Delhi", "Mumbai", "Bengaluru"]))
        .then(pl.col("delivery_min") > 15)
        .otherwise(pl.col("delivery_min") > 20)
        .alias("is_late")
    )
)
print(f"Time taken by polars: {polars_ms:.2f} ms")

results.append([task, pandas_apply_ms, pandas_vector_ms, numpy_ms, polars_ms])


Task: late_delivery_flag
Time taken by pandas_apply: 311.87 ms
Time taken by pandas_vector: 2.54 ms
Time taken by numpy: 1.88 ms
Time taken by polars: 0.54 ms


In [5]:
task = "Basket tier from items"
print(f"Task: {task}")

# Pandas Apply Function
orders = pd.read_csv("../data/orders.csv", parse_dates=["order_ts"])

assert pd.api.types.is_integer_dtype(orders["items"]) and (orders["items"] >= 1).all(), \
    "basket tiers assume integer items >= 1"

pandas_apply_ms = best_ms(
    lambda: orders.apply(
        lambda x:
        "mini" if x["items"] <= 2
        else "regular" if x["items"] <= 5
        else "large" if x["items"] <= 9
        else "stock_up",
        axis = 1
    )
)
print(f"Time taken by pandas_apply: {pandas_apply_ms:.2f} ms")

# Pandas Vector Function
orders = pd.read_csv("../data/orders.csv", parse_dates=["order_ts"])

assert pd.api.types.is_integer_dtype(orders["items"]) and (orders["items"] >= 1).all(), \
    "basket tiers assume integer items >= 1"

pandas_vector_ms = best_ms(
    lambda: pd.cut(
        orders["items"],
        bins = [0, 2, 5, 9, np.inf],
        labels = ["mini", "regular", "large", "stock_up"]
    )
)
print(f"Time taken by pandas_vector: {pandas_vector_ms:.2f} ms")

# Numpy Function
orders = pd.read_csv("../data/orders.csv", parse_dates=["order_ts"])

assert pd.api.types.is_integer_dtype(orders["items"]) and (orders["items"] >= 1).all(), \
    "basket tiers assume integer items >= 1"

numpy_ms = best_ms(
    lambda: np.select(
        [orders["items"] <= 2, orders["items"] <= 5, orders["items"] <= 9],
        ["mini", "regular", "large"],
        default = "stock_up"
    )
)
print(f"Time taken by numpy: {numpy_ms:.2f} ms")

# Polars Function
orders = pl.read_csv("../data/orders.csv", try_parse_dates=True)
polars_ms = best_ms(
    lambda: orders.with_columns(
        pl.when(pl.col("items") <= 2).then(pl.lit("mini"))
        .when(pl.col("items") <= 5).then(pl.lit("regular"))
        .when(pl.col("items") <= 9).then(pl.lit("large"))
        .otherwise(pl.lit("stock_up"))
        .alias("basket_tier")
    )
)
print(f"Time taken by polars: {polars_ms:.2f} ms")

results.append([task, pandas_apply_ms, pandas_vector_ms, numpy_ms, polars_ms])


Task: Basket tier from items
Time taken by pandas_apply: 316.59 ms
Time taken by pandas_vector: 1.33 ms
Time taken by numpy: 1.46 ms
Time taken by polars: 0.62 ms


In [6]:
task = "days_since_prev_order"
print(f"Task: {task}")

# Pandas Apply Function
orders = pd.read_csv("../data/orders.csv", parse_dates=["order_ts"])
pandas_apply_ms = best_ms(
    lambda: (orders["order_ts"] - orders.groupby("user_id", group_keys=False)["order_ts"].apply(lambda x: x.shift(1)))/pd.Timedelta(days=1)
)
print(f"Time taken by pandas_apply: {pandas_apply_ms:.2f} ms")

# Pandas Vector Function
orders = pd.read_csv("../data/orders.csv", parse_dates=["order_ts"])
pandas_vendor_ms = best_ms(
    lambda: orders["order_ts"].groupby(orders["user_id"]).diff() / pd.Timedelta(days=1)
)
print(f"Time taken by pandas_vector: {pandas_vendor_ms:.2f} ms")

# Numpy Function
orders = pd.read_csv("../data/orders.csv", parse_dates=["order_ts"])
numpy_ms = best_ms(
    lambda: np.where(
        np.concatenate(([np.nan], orders["user_id"][:-1])) == orders["user_id"],
        (orders["order_ts"] - np.concatenate((np.array(["NaT"], dtype=orders["order_ts"].to_numpy().dtype), orders["order_ts"][:-1])))/pd.Timedelta(days=1),
        np.nan
    )
)
print(f"Time taken by numpy: {numpy_ms:.2f} ms")

# Polars Function
orders = pl.read_csv("../data/orders.csv", try_parse_dates=True)
polars_ms = best_ms(
    lambda: orders.sort(["user_id", "order_ts"])
    .with_columns(
        (pl.col("order_ts").diff().over("user_id") / pl.duration(days=1))
        .alias("days_since_prev")
    )
)
print(f"Time taken by polars: {polars_ms:.2f} ms")

results.append([task, pandas_apply_ms, pandas_vector_ms, numpy_ms, polars_ms])


Task: days_since_prev_order
Time taken by pandas_apply: 677.34 ms
Time taken by pandas_vector: 4.68 ms
Time taken by numpy: 0.88 ms
Time taken by polars: 30.05 ms


In [7]:
results_df = pd.DataFrame(
    results,
    columns = ["task", "pandas_apply_ms", "pandas_vector_ms", "numpy_ms", "polars_ms"]
)

results_df["vector_speedup"] = results_df["pandas_apply_ms"]/results_df["pandas_vector_ms"]
results_df["numpy_speedup"] = results_df["pandas_apply_ms"]/results_df["numpy_ms"]
results_df["polars_speedup"] = results_df["pandas_apply_ms"]/results_df["polars_ms"]

results_df

,task,pandas_apply_ms,pandas_vector_ms,numpy_ms,polars_ms,vector_speedup,numpy_speedup,polars_speedup
0,late_delivery_flag,311.8656,2.5375,1.8781,0.5446,122.902700,166.053777,572.650769
1,Basket tier from items,316.5889,1.3314,1.4579,0.6197,237.786464,217.154060,510.874452
2,days_since_prev_order,677.3351,1.3314,0.8808,30.0472,508.738993,768.999875,22.542370


Polars has better speed in everywhere except `days_since_prev_order`, so for windows functions numpy has better speed